# Use Lasso to predict treatment (binary), then see which genes are most important to predict

In [1]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("LASSO_bootstrap").getOrCreate()
sc = spark.sparkContext
print(sc.defaultParallelism)

4


26/05/30 21:10:44 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [2]:
import pandas as pd
import numpy as np
import pyspark
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("LASSO_bootstrap").getOrCreate()
sc = spark.sparkContext

cpm = pd.read_parquet('gs://gene_datasets/sample_by_gene.parquet')
pca_summary = pd.read_csv('gs://gene_datasets/pca_summary.csv')

pareto_treatments = pca_summary.loc[pca_summary['pareto_optimal'], 'sample_type'].tolist()
print(f"Pareto-optimal treatments ({len(pareto_treatments)}): {pareto_treatments}")

Pareto-optimal treatments (4): ['hATF567', 'hATF561', 'nZF105', 'nZF139']


/opt/conda/miniconda3/lib/python3.10/site-packages/google/api_core/_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.8) which Google will stop supporting in new releases of google.cloud.storage_control_v2 once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.cloud.storage_control_v2 past that date.
  warnings.warn(message, FutureWarning)


In [3]:
# Filter to pareto treatments + Control
mask = cpm['sample_type'].isin(pareto_treatments + ['Control'])
sub = cpm[mask].copy()
gene_cols = [c for c in sub.columns if c.startswith('ENSG')]

print(f"Samples: {len(sub)}, Genes: {len(gene_cols)}")
print(sub['sample_type'].value_counts())

X_scaled = StandardScaler().fit_transform(sub[gene_cols].values)
y = (sub['sample_type'] != 'Control').astype(int).values

assert (y == 0).any() and (y == 1).any(), "y must contain both classes before bootstrapping"

X_broadcast = sc.broadcast(X_scaled)
y_broadcast = sc.broadcast(y)

def fit_one_bootstrap(seed):
    import numpy as np
    from sklearn.linear_model import LogisticRegression
    
    X, y_local = X_broadcast.value, y_broadcast.value
    rng = np.random.RandomState(seed)
    
    treat_idx = np.where(y_local == 1)[0]
    control_idx = np.where(y_local == 0)[0]

    if len(treat_idx) == 0 or len(control_idx) == 0:
        return [0] * X.shape[1]
    
    boot_treat = rng.choice(treat_idx, len(treat_idx), replace=True)
    boot_control = rng.choice(control_idx, len(control_idx), replace=True)
    
    idx = np.concatenate([boot_treat, boot_control])
    y_boot = y_local[idx]

    if len(np.unique(y_boot)) < 2:
        return [0] * X.shape[1]
    
    lasso = LogisticRegression(
        penalty='l1', solver='liblinear', C=10.0, max_iter=5000
    )
    lasso.fit(X[idx], y_boot)
    return (lasso.coef_[0] != 0).astype(int).tolist()

N_BOOTSTRAP = 50
seeds_rdd = sc.parallelize(range(N_BOOTSTRAP), numSlices=10)
selection_results = seeds_rdd.map(fit_one_bootstrap).collect()

selection_freq = np.array(selection_results).sum(axis=0) / N_BOOTSTRAP

# Show distribution of selection frequencies before filtering
freq_series = pd.Series(selection_freq, index=gene_cols).sort_values(ascending=False)
print("\nSelection frequency distribution (genes selected in >0% of bootstraps):")
print(freq_series[freq_series > 0].describe())
print(f"\nTop 10 genes by selection frequency:")
print(freq_series.head(10))

stable_genes = pd.DataFrame({
    'gene_id': gene_cols,
    'selection_frequency': selection_freq,
}).query('selection_frequency >= 0.5').sort_values('selection_frequency', ascending=False)

print(f"\nStably selected genes (>=50% of bootstraps): {len(stable_genes)}")
print(stable_genes.head(30))

Samples: 10, Genes: 78893
sample_type
hATF561     2
hATF567     2
nZF105      2
nZF139      2
Control     2
Base        0
nZF153      0
nZF93       0
nZF81       0
nZF42       0
nZF36       0
nZF156      0
nZF154      0
nZF147      0
nZF151      0
nZF148      0
nZF145      0
hATF555R    0
hATF555Q    0
ZDS2        0
SP1R        0
nZFD96      0
Name: count, dtype: int64



Selection frequency distribution (genes selected in >0% of bootstraps):
count    3289.000000
mean        0.025746
std         0.012240
min         0.020000
25%         0.020000
50%         0.020000
75%         0.020000
max         0.100000
dtype: float64

Top 10 genes by selection frequency:
ENSG00000099256    0.10
ENSG00000268870    0.10
ENSG00000179299    0.10
ENSG00000272312    0.10
ENSG00000219433    0.08
ENSG00000237409    0.08
ENSG00000236018    0.08
ENSG00000183751    0.08
ENSG00000137656    0.08
ENSG00000306255    0.08
dtype: float64

Stably selected genes (>=50% of bootstraps): 0
Empty DataFrame
Columns: [gene_id, selection_frequency]
Index: []


We applied bootstrap-stabilized LASSO to identify genes distinguishing Pareto-optimal treatments from control. No gene was selected in more than 6% of bootstraps, indicating that no single gene reliably discriminates treatment status given the available sample size. Instead, treatment effects are diffuse across many weakly-discriminating genes, consistent with the broad transcriptomic perturbation observed in PCA.

# Problem with lasso approach (keep for showing trial and error and pivoting)
essentially the issue here is that Lasso, even with bootstrapping only has 3 options with the 2 controls. We have (C1, C2), (C1, C1), (C2, C2). This is a really small sample space and so we fail to accurately model variance, so the confidence intervals around the Lasso coefficients will be incredibly tight.

Second, only two samples cannot capture the natural variance in the control population. In an 80k dimensional space the probably that some genes will randomly have exteme values in those two specific samples compared to the 22 treatments is kinda high

Lastly, the 22-2 ratio is really imbalanced, and lasso can only handle at most n variables before it gets really saturated so we can at most only select 24/80k. 

# Solution to Lasso problem (maybe?)




In [4]:
# Get unique gene IDs that have any Open Targets association
disease_associations = pd.read_parquet('gs://gene_datasets/disease_associations.parquet')
clinically_relevant_genes = set(disease_associations['gene_id'].unique())
print(f"Genes with Open Targets associations: {len(clinically_relevant_genes)}")

# Filter your gene matrix to just those
filtered_gene_cols = [g for g in gene_cols if g in clinically_relevant_genes]
print(f"Filtered from {len(gene_cols)} to {len(filtered_gene_cols)} genes")

# Build filtered X, scale, broadcast
X_filtered = sub[filtered_gene_cols].values
X_filtered_scaled = StandardScaler().fit_transform(X_filtered)

X_broadcast = sc.broadcast(X_filtered_scaled)  # overwrite previous broadcast
# y_broadcast is unchanged

# Update gene_cols to the filtered list so downstream uses the right names
gene_cols = filtered_gene_cols

Genes with Open Targets associations: 18845
Filtered from 78893 to 18845 genes


In [5]:
def fit_one_nsc_bootstrap(seed):
    import numpy as np
    from sklearn.neighbors import NearestCentroid
    
    X, y_local = X_broadcast.value, y_broadcast.value
    rng = np.random.RandomState(seed)
    
    treat_idx = np.where(y_local == 1)[0]
    control_idx = np.where(y_local == 0)[0]
    idx = np.concatenate([
        rng.choice(treat_idx, len(treat_idx), replace=True),
        rng.choice(control_idx, len(control_idx), replace=True),
    ])
    
    nsc = NearestCentroid(shrink_threshold=THRESHOLD_VAL)
    nsc.fit(X[idx], y_local[idx])
    
    # Genes with non-zero centroid difference between classes are "selected"
    diff = nsc.centroids_[1] - nsc.centroids_[0]
    return (diff != 0).astype(int).tolist()

N_BOOTSTRAP = 50
seeds_rdd = sc.parallelize(range(N_BOOTSTRAP), numSlices=10)

THRESHOLD = [0.5, 1.0, 1.5, 2.0] # test some different levels

for THRESHOLD_VAL in THRESHOLD:
    print("Testing value:", THRESHOLD_VAL)
    selection_results = seeds_rdd.map(fit_one_nsc_bootstrap).collect()

    selection_freq = np.array(selection_results).sum(axis=0) / N_BOOTSTRAP

    nsc_stability = pd.DataFrame({
        'gene_id': gene_cols,
        'selection_frequency': selection_freq,
    })

    print("Selection frequency distribution:")
    print(nsc_stability['selection_frequency'].describe())
    print(f"\nGenes selected in >=50% of bootstraps: {(nsc_stability['selection_frequency'] >= 0.5).sum()}")
    print(f"Genes selected in >=70% of bootstraps: {(nsc_stability['selection_frequency'] >= 0.7).sum()}")
    print(f"Genes selected in 100% of bootstraps: {(nsc_stability['selection_frequency'] == 1.0).sum()}")

    print("\nTop 20 stable genes:")
    print(nsc_stability.nlargest(20, 'selection_frequency'))

Testing value: 0.5


Selection frequency distribution:
count    18845.000000
mean         0.481782
std          0.320983
min          0.000000
25%          0.200000
50%          0.500000
75%          0.740000
max          1.000000
Name: selection_frequency, dtype: float64

Genes selected in >=50% of bootstraps: 9499
Genes selected in >=70% of bootstraps: 5389
Genes selected in 100% of bootstraps: 1484

Top 20 stable genes:
             gene_id  selection_frequency
3    ENSG00000000460                  1.0
15   ENSG00000001629                  1.0
17   ENSG00000001631                  1.0
20   ENSG00000002549                  1.0
31   ENSG00000003096                  1.0
37   ENSG00000003402                  1.0
63   ENSG00000004864                  1.0
65   ENSG00000004897                  1.0
69   ENSG00000004975                  1.0
72   ENSG00000005020                  1.0
95   ENSG00000005471                  1.0
96   ENSG00000005483                  1.0
99   ENSG00000005700                  1.0
101  E

Selection frequency distribution:
count    18845.000000
mean         0.225824
std          0.265468
min          0.000000
25%          0.000000
50%          0.120000
75%          0.360000
max          1.000000
Name: selection_frequency, dtype: float64

Genes selected in >=50% of bootstraps: 2863
Genes selected in >=70% of bootstraps: 1640
Genes selected in 100% of bootstraps: 392

Top 20 stable genes:
             gene_id  selection_frequency
20   ENSG00000002549                  1.0
95   ENSG00000005471                  1.0
96   ENSG00000005483                  1.0
99   ENSG00000005700                  1.0
206  ENSG00000008294                  1.0
228  ENSG00000009694                  1.0
353  ENSG00000015479                  1.0
416  ENSG00000023287                  1.0
431  ENSG00000024526                  1.0
433  ENSG00000025039                  1.0
487  ENSG00000033030                  1.0
515  ENSG00000035928                  1.0
538  ENSG00000038295                  1.0
613  EN

Selection frequency distribution:
count    18845.000000
mean         0.095135
std          0.165538
min          0.000000
25%          0.000000
50%          0.020000
75%          0.120000
max          1.000000
Name: selection_frequency, dtype: float64

Genes selected in >=50% of bootstraps: 598
Genes selected in >=70% of bootstraps: 308
Genes selected in 100% of bootstraps: 96

Top 20 stable genes:
              gene_id  selection_frequency
99    ENSG00000005700                  1.0
206   ENSG00000008294                  1.0
613   ENSG00000047597                  1.0
739   ENSG00000055732                  1.0
861   ENSG00000064218                  1.0
930   ENSG00000065809                  1.0
1017  ENSG00000068137                  1.0
1027  ENSG00000068489                  1.0
1316  ENSG00000076108                  1.0
1412  ENSG00000078674                  1.0
1469  ENSG00000080298                  1.0
1561  ENSG00000083093                  1.0
1831  ENSG00000089916                  

Selection frequency distribution:
count    18845.000000
mean         0.039989
std          0.097956
min          0.000000
25%          0.000000
50%          0.000000
75%          0.020000
max          1.000000
Name: selection_frequency, dtype: float64

Genes selected in >=50% of bootstraps: 89
Genes selected in >=70% of bootstraps: 51
Genes selected in 100% of bootstraps: 18

Top 20 stable genes:
               gene_id  selection_frequency
613    ENSG00000047597                 1.00
739    ENSG00000055732                 1.00
861    ENSG00000064218                 1.00
3475   ENSG00000108576                 1.00
4264   ENSG00000115085                 1.00
4761   ENSG00000119004                 1.00
4789   ENSG00000119397                 1.00
6382   ENSG00000132446                 1.00
6716   ENSG00000134627                 1.00
7712   ENSG00000140386                 1.00
8022   ENSG00000142687                 1.00
10156  ENSG00000160551                 1.00
11855  ENSG00000168135      

tuned threshold in NSC without cv because using any sort of CV might just not have controls in there, and 11 fold CV doesnt work becuase we will have a fold with no controls so we just need to experiment

In [6]:
# Updated bootstrap function that takes (seed, threshold) tuple
def fit_one_nsc_bootstrap_with_threshold(seed_and_threshold):
    import numpy as np
    from sklearn.neighbors import NearestCentroid
    
    seed, threshold = seed_and_threshold
    X, y_local = X_broadcast.value, y_broadcast.value
    rng = np.random.RandomState(seed)
    treat_idx = np.where(y_local == 1)[0]
    control_idx = np.where(y_local == 0)[0]
    idx = np.concatenate([
        rng.choice(treat_idx, len(treat_idx), replace=True),
        rng.choice(control_idx, len(control_idx), replace=True),
    ])
    
    nsc = NearestCentroid(shrink_threshold=threshold)
    nsc.fit(X[idx], y_local[idx])
    diff = nsc.centroids_[1] - nsc.centroids_[0]
    return (threshold, (diff != 0).astype(int).tolist())

THRESHOLDS = [0.5, 1.0, 1.5, 2.0, 2.5, 3.0]
N_BOOTSTRAP = 50

# Build all (seed, threshold) jobs and run in parallel in one shot
all_jobs = [(seed, t) for t in THRESHOLDS for seed in range(N_BOOTSTRAP)]
jobs_rdd = sc.parallelize(all_jobs, numSlices=20)
all_results = jobs_rdd.map(fit_one_nsc_bootstrap_with_threshold).collect()

# Aggregate per threshold, save stable gene sets for intersection
threshold_stable_sets = {}
threshold_stability_dfs = {}

for threshold in THRESHOLDS:
    selections = [r[1] for r in all_results if r[0] == threshold]
    freq = np.array(selections).sum(axis=0) / N_BOOTSTRAP
    
    nsc_stability = pd.DataFrame({
        'gene_id': gene_cols,
        'selection_frequency': freq,
    })
    threshold_stability_dfs[threshold] = nsc_stability
    
    # Save the 100%-stable gene set for the intersection check
    threshold_stable_sets[threshold] = set(nsc_stability.query('selection_frequency == 1.0')['gene_id'])
    
    n_at_100 = (freq == 1.0).sum()
    n_at_70 = (freq >= 0.7).sum()
    n_at_50 = (freq >= 0.5).sum()
    print(f"Threshold {threshold}: {n_at_100} at 100%, {n_at_70} at >=70%, {n_at_50} at >=50%")

# Robustness check: genes 100%-stable across ALL thresholds tested
robust_genes = set.intersection(*threshold_stable_sets.values())
print(f"\nGenes 100%-stable across ALL tested thresholds: {len(robust_genes)}")

# Also check the more practical intersection: stable at strict thresholds only
# (drop threshold 0.5 since it's typically too permissive to be meaningful)
strict_robust = set.intersection(*[threshold_stable_sets[t] for t in THRESHOLDS if t <= 2.5])
print(f"Genes 100%-stable across thresholds (excluding the 3.0): {len(strict_robust)}")

print(f"\nRobust gene IDs (excluding the 3.0):")
print(sorted(strict_robust))

Threshold 0.5: 1484 at 100%, 5389 at >=70%, 9499 at >=50%
Threshold 1.0: 392 at 100%, 1640 at >=70%, 2863 at >=50%
Threshold 1.5: 96 at 100%, 308 at >=70%, 598 at >=50%
Threshold 2.0: 18 at 100%, 51 at >=70%, 89 at >=50%
Threshold 2.5: 7 at 100%, 7 at >=70%, 14 at >=50%
Threshold 3.0: 0 at 100%, 5 at >=70%, 7 at >=50%

Genes 100%-stable across ALL tested thresholds: 0
Genes 100%-stable across thresholds (excluding the 3.0): 7

Robust gene IDs (excluding the 3.0):
['ENSG00000047597', 'ENSG00000064218', 'ENSG00000108576', 'ENSG00000115085', 'ENSG00000132446', 'ENSG00000134627', 'ENSG00000168135']


In [7]:
def fit_one_nsc_permuted(args):
    import numpy as np
    from sklearn.neighbors import NearestCentroid
    
    seed, threshold = args
    X = X_broadcast.value
    y_true = y_broadcast.value
    
    rng = np.random.RandomState(seed)
    y_permuted = rng.permutation(y_true)
    
    nsc = NearestCentroid(shrink_threshold=threshold)
    nsc.fit(X, y_permuted)
    diff = nsc.centroids_[1] - nsc.centroids_[0]
    return (threshold, (diff != 0).sum())  # just count, don't track which genes

N_PERMUTATIONS = 100
permutation_jobs = [(seed, t) for t in THRESHOLDS for seed in range(N_PERMUTATIONS)]
perm_rdd = sc.parallelize(permutation_jobs, numSlices=20)
perm_results = perm_rdd.map(fit_one_nsc_permuted).collect()

print(f"\n{'Threshold':<10} {'Real (any stability)':<22} {'Random mean ± std':<22} {'P-value approx':<15}")
for threshold in THRESHOLDS:
    perm_counts = [r[1] for r in perm_results if r[0] == threshold]
    # Your real selection counts at this threshold (use 50% stability for "selected")
    real_count = (threshold_stability_dfs[threshold]['selection_frequency'] >= 0.5).sum()
    perm_mean = np.mean(perm_counts)
    perm_std = np.std(perm_counts)
    # Approx p-value: fraction of permutations that got at least as many genes
    p = np.mean([c >= real_count for c in perm_counts])
    print(f"{threshold:<10} {real_count:<22} {perm_mean:.0f} ± {perm_std:.0f}{'':<10} {p:.3f}")


Threshold  Real (any stability)   Random mean ± std      P-value approx 
0.5        9499                   5565 ± 821           0.000
1.0        2863                   1045 ± 367           0.000
1.5        598                    79 ± 51           0.000
2.0        89                     12 ± 5           0.000
2.5        14                     8 ± 3           0.030
3.0        7                      0 ± 0           0.000


In [8]:
REFINE_THRESHOLDS = [2.0, 2.1, 2.2, 2.3, 2.4, 2.5]
N_BOOTSTRAP = 50
N_PERMUTATIONS = 100

# Step 1 — bootstrap stability at the new thresholds
boot_jobs = [(seed, t) for t in REFINE_THRESHOLDS for seed in range(N_BOOTSTRAP)]
boot_rdd = sc.parallelize(boot_jobs, numSlices=20)
boot_results = boot_rdd.map(fit_one_nsc_bootstrap_with_threshold).collect()

# Compute real selection counts per threshold
real_counts = {}
for threshold in REFINE_THRESHOLDS:
    selections = [r[1] for r in boot_results if r[0] == threshold]
    freq = np.array(selections).sum(axis=0) / N_BOOTSTRAP
    real_counts[threshold] = (freq >= 0.5).sum()  # genes at >=50% stability

# Step 2 — permutation null at the new thresholds
perm_jobs = [(seed, t) for t in REFINE_THRESHOLDS for seed in range(N_PERMUTATIONS)]
perm_rdd = sc.parallelize(perm_jobs, numSlices=20)
perm_results = perm_rdd.map(fit_one_nsc_permuted).collect()

# Step 3 — report
print(f"\n{'Threshold':<12} {'Real (>=50% stable)':<22} {'Random mean ± std':<22} {'P-value':<10}")
for threshold in REFINE_THRESHOLDS:
    perm_counts = [r[1] for r in perm_results if r[0] == threshold]
    perm_mean = np.mean(perm_counts)
    perm_std = np.std(perm_counts)
    p = np.mean([c >= real_counts[threshold] for c in perm_counts])
    print(f"{threshold:<12} {real_counts[threshold]:<22} {perm_mean:.0f} ± {perm_std:.0f}{'':<14} {p:.3f}")


Threshold    Real (>=50% stable)    Random mean ± std      P-value   
2.0          89                     12 ± 5               0.000
2.1          66                     10 ± 4               0.000
2.2          44                     8 ± 4               0.000
2.3          30                     8 ± 3               0.000
2.4          22                     8 ± 3               0.000
2.5          14                     8 ± 3               0.030


We selected threshold 2.2 as the most stringent threshold maintaining strong significance (p < 0.001) against the permutation null. This yielded 19 genes selected in ≥50% of bootstraps. We also report results at threshold 2.0 (46 genes, p < 0.001) as a more inclusive view for sensitivity analysis."

In [9]:
freq_24 = np.array([r[1] for r in boot_results if r[0] == 2.4]).sum(axis=0) / N_BOOTSTRAP
threshold_stability_dfs[2.4] = pd.DataFrame({
    'gene_id': gene_cols,
    'selection_frequency': freq_24,
})

for threshold in [2.0, 2.4]:
    stable = (threshold_stability_dfs[threshold]
              .query('selection_frequency >= 0.5')
              .sort_values('selection_frequency', ascending=False))
    print(f"\n{'='*60}")
    print(f"THRESHOLD {threshold}: {len(stable)} genes at >=50% stability")
    print(f"{'='*60}")
    print(stable.to_string(index=False))


THRESHOLD 2.0: 89 genes at >=50% stability
        gene_id  selection_frequency
ENSG00000132446                 1.00
ENSG00000189350                 1.00
ENSG00000188817                 1.00
ENSG00000168135                 1.00
ENSG00000134627                 1.00
ENSG00000140386                 1.00
ENSG00000214029                 1.00
ENSG00000115085                 1.00
ENSG00000160551                 1.00
ENSG00000119004                 1.00
ENSG00000119397                 1.00
ENSG00000178295                 1.00
ENSG00000177669                 1.00
ENSG00000142687                 1.00
ENSG00000064218                 1.00
ENSG00000055732                 1.00
ENSG00000047597                 1.00
ENSG00000108576                 1.00
ENSG00000133958                 0.98
ENSG00000276368                 0.98
ENSG00000240720                 0.98
ENSG00000138758                 0.98
ENSG00000151338                 0.98
ENSG00000119421                 0.96
ENSG00000107362                

The actual computation you ran was:

6 thresholds (initial) + 6 thresholds (refinement) = 12 thresholds
50 bootstraps + 100 permutations per threshold = 150 iterations per threshold
12 × 150 = 1,800 independent NSC fits
Each fit is small (NSC is fast), but 1,800 of them is enough that single-threaded execution would take noticeable time and benefit from parallelization. This is the canonical "embarrassingly parallel" workload Spark exists for — many independent jobs with no cross-talk needed between workers.

# Now Enrich these results with opentargets

In [10]:
# Pick your stable gene set (using threshold 2.2 as primary)
PRIMARY_THRESHOLD = 2.4
stable_genes = (threshold_stability_dfs[PRIMARY_THRESHOLD]
                .query('selection_frequency >= 0.5'))

# Enrich
nsc_enriched = stable_genes.merge(
    disease_associations, on='gene_id', how='left'
)

print(f"Stable genes: {len(stable_genes)}")
print(f"Genes with at least one disease association: "
      f"{nsc_enriched.dropna(subset=['disease_name'])['gene_id'].nunique()}")

# === Top diseases linked to stable genes ===
top_diseases = (nsc_enriched
                .dropna(subset=['disease_name'])
                .groupby('disease_name')
                .agg(n_genes=('gene_id', 'nunique'),
                     avg_score=('association_score', 'mean'),
                     max_score=('association_score', 'max'))
                .query('n_genes >= 2')  # at least 2 stable genes converge on the trait
                .sort_values(['n_genes', 'avg_score'], ascending=False)
                .head(20))

# === Top therapeutic areas ===
exploded = (nsc_enriched
            .dropna(subset=['disease_name'])
            .explode('therapeutic_areas')
            .dropna(subset=['therapeutic_areas']))

top_areas = (exploded
             .groupby('therapeutic_areas')
             .agg(n_genes=('gene_id', 'nunique'),
                  n_diseases=('disease_name', 'nunique'),
                  avg_score=('association_score', 'mean'))
             .sort_values('n_genes', ascending=False))


# === Drill into the 3 ultra-robust genes ===
ultra_robust_ids = ['ENSG00000047597', 'ENSG00000064218', 'ENSG00000132446']
ultra_robust_details = (nsc_enriched[nsc_enriched['gene_id'].isin(ultra_robust_ids)]
                        .dropna(subset=['disease_name'])
                        .sort_values(['gene_id', 'association_score'], ascending=[True, False]))

# Save final results to GCS
#stable_genes.to_csv('gs://gene_datasets/nsc_stable_genes_t22.csv', index=False)
#top_diseases.to_csv('gs://gene_datasets/nsc_top_diseases.csv')
#top_areas.to_csv('gs://gene_datasets/nsc_top_areas.csv')
#print("\nSaved results to gs://gene_datasets/")

Stable genes: 22
Genes with at least one disease association: 22


In [11]:
print("\nDiseases linked to stable genes:")
top_diseases


Diseases linked to stable genes:


,n_genes,avg_score,max_score
disease_name,,,
neurodegenerative disease,10,0.364820,0.556603
body height,8,0.317413,0.511564
platelet volume,7,0.357856,0.521286
erythrocyte count,6,0.370091,0.506992
body mass index,6,0.362208,0.514759
sitting height measurement,6,0.348904,0.499292
eosinophil count,6,0.338996,0.538651
serum creatinine amount,6,0.297800,0.475605
Red cell distribution width,6,0.285197,0.355384


In [12]:
print("\nTherapeutic areas affected by stable genes:")
top_areas


Therapeutic areas affected by stable genes:


,n_genes,n_diseases,avg_score
therapeutic_areas,,,
measurement,19,207,0.280034
nervous system disease,18,70,0.353967
phenotype,13,69,0.331464
"genetic, familial or congenital disease",12,40,0.348303
reproductive system or breast disease,11,17,0.239081
cancer or benign tumor,11,21,0.231605
musculoskeletal or connective tissue disease,10,21,0.259994
psychiatric disorder,8,37,0.372303
gastrointestinal disease,8,17,0.261097


In [13]:
print("\n3 ultra-robust genes — top associations:")
print(ultra_robust_details.head(30).to_string(index=False))


3 ultra-robust genes — top associations:
        gene_id  selection_frequency                                              disease_name    disease_id                                                                                                          therapeutic_area_ids  association_score                                                                                               therapeutic_areas
ENSG00000047597                  1.0                       McLeod neuroacanthocytosis syndrome MONDO_0018945 {'list': [{'element': 'EFO_0000618'}, {'element': 'OTAR_0000018'}, {'element': 'MONDO_0002025'}, {'element': 'EFO_0000651'}]}           0.804145              [nervous system disease, genetic, familial or congenital disease, psychiatric disorder, phenotype]
ENSG00000047597                  1.0                                          genetic disorder   EFO_0000508                                                                                       {'list': [{'element': 'OTAR_000

NSC is like which genes, taken together, allow me to correctly classify a sample as treatment vs control?

1. A gene that changes a lot but doesn't help predict.
Suppose Gene X has expression: Control = [10, 2], Treatment = [3, 11, 1, 14, 0, 13, 2, 12, 4, 10]. The mean difference is small (~6), so log2FC flags it as "changed." But the spread within treatments is huge — knowing Gene X's value doesn't help you predict if a sample is treatment or control, because both classes have values across the whole range. NSC would not select this gene.
2. A gene that didn't change much but reliably predicts.
Gene Y expression: Control = [5.0, 5.1], Treatment = [4.0, 4.0, 4.1, 4.0, 4.1, 4.0, 4.0, 4.1, 4.0, 4.0]. The fold change is small (~20%), but the means are cleanly separated with low within-class variance. Knowing Gene Y's value perfectly predicts class. NSC would select it; log2FC might not even flag it as interesting.
3. Correlated genes — the most important difference.
If 10 genes all change identically because they're in the same biological pathway, log2FC flags all 10. NSC and especially LASSO might keep just 1 of them — because once you have one, the other 9 contain no additional predictive information.

gene expression data plotted in space with each gene being a seperate axis, we have that cloud of treatment samples and cloud of control samples. For each of these classes, compute the average location of all its samples, which gives us the centroid. For a new sample, we see which centroid it is closer to. Thus the centroids would be computed using all 80k genes but the shrinkage fixes that.

Compute the global mean per gene. Average expression across all samples ignoring class. Then we compute each class's mean per gene: For gene X, the average expression among treatment samples, average among control samples. These become the class centroids. Then we compute the class-specific deviation per gene. This is basically a per-gene t-statistics which says how far the treatment class mean is from the overall mean, normalized by how noisy the gene is. A big deviation will mean the class mean is far from the overall mean relative to noise, so the gene meaninfully distinguishes the class.

Small deviation means the class mean is similar to the overall mean or the deviation might just be variation in the measurement. 

Then we shrink the deviations towards 0, using some shrink threshold, for each class, gene deviation, we shrink. Basically, if a deviation is smaller than the threshold it gets shrunk to exactly zero. If its larger than the threshold, it gets pulled towards 0 by delta. 

Now for each gene that the shrunken deviation is 0, its shrunken centroid will be the overall mean and so that gene will contribute nothing to the classification. 

Genes selected by the algorithm are the ones where at least one class's deviation survived the shrinkage and didnt get 0d out. 

A new sample would be classified by euclidean distance to the shrunken centroids.

In [19]:
import plotly.express as px
from sklearn.decomposition import PCA

# Run PCA on the filtered+scaled gene matrix
pca = PCA(n_components=2)
pcs = pca.fit_transform(X_filtered_scaled)
sub = sub.reset_index()

plot_df = pd.DataFrame({
    'PC1': pcs[:, 0],
    'PC2': pcs[:, 1],
    'sample': sub['sample'].values,
    'group': sub['sample_type'].values,
    'is_control': sub['sample_type'].values == 'Control',
})

fig = px.scatter(
    plot_df, x='PC1', y='PC2', color='group', hover_name='sample',
    title=f'PCA of all samples — PC1 ({pca.explained_variance_ratio_[0]:.1%}), PC2 ({pca.explained_variance_ratio_[1]:.1%})',
    width=900, height=600,
)
fig.show()

In [21]:
import plotly.graph_objects as go

# Derive stable gene IDs and their expression matrix from existing variables
stable_gene_ids = (threshold_stability_dfs[PRIMARY_THRESHOLD]
                   .query('selection_frequency >= 0.5')
                   .sort_values('selection_frequency', ascending=False)['gene_id']
                   .tolist())

stable_col_idx = [gene_cols.index(g) for g in stable_gene_ids]
X_stable = X_filtered_scaled[:, stable_col_idx]

# Build heatmap dataframe: genes as rows, samples as columns
heatmap_df = pd.DataFrame(X_stable, index=sub['sample'].values, columns=stable_gene_ids).T

# Sort samples so controls are grouped together visually
sample_order = sub.sort_values('sample_type')['sample'].tolist()
heatmap_df = heatmap_df[sample_order]

# Build sample-type annotation for x-axis tick labels
sample_type_map = sub.set_index('sample')['sample_type'].to_dict()
x_labels = [f"{s}<br><i>{sample_type_map[s]}</i>" for s in heatmap_df.columns]

fig = go.Figure(go.Heatmap(
    z=heatmap_df.values,
    x=x_labels,
    y=heatmap_df.index,
    colorscale='RdBu_r',
    zmid=0,
    colorbar=dict(title='Standardized<br>expression'),
))
fig.update_layout(
    title=f'NSC-Selected Gene Expression Across Samples (threshold={PRIMARY_THRESHOLD}, {len(stable_gene_ids)} genes)',
    xaxis_title='Sample',
    yaxis_title='Gene',
    xaxis=dict(tickangle=-45),
    height=700, width=950,
    template='plotly_white',
)
fig.show()


The values are standardized expression (z-scores across samples, per gene). So for each gene:

Red = expression in that sample is above the gene's average across all samples
Blue = expression in that sample is below the gene's average across all samples
White = expression is right at the gene's average

# Broad-signature analysis: 1,400-gene NSC set (threshold=0.5, 100% bootstrap stability)

In [22]:
# --- Setup: extract the 1,400-gene broad signature ---
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from scipy import stats

# 1,400 genes selected at threshold=0.5 with 100% bootstrap stability
broad_sig_df = threshold_stability_dfs[0.5].query('selection_frequency == 1.0').copy()
broad_gene_ids = broad_sig_df['gene_id'].tolist()
print(f"Broad signature genes: {len(broad_gene_ids)}")

# Index into X_filtered_scaled (column order matches gene_cols, which is filtered_gene_cols)
broad_col_idx = [gene_cols.index(g) for g in broad_gene_ids]
X_broad = X_filtered_scaled[:, broad_col_idx]  # (10 samples) × (1400 genes)

# Raw CPM expression for log2FC computation (unscaled)
X_broad_raw = sub[broad_gene_ids].values.astype(float)

# Class labels
is_control = (sub['sample_type'] == 'Control').values
is_treatment = ~is_control

# --- Compute log2FC (treatment mean vs control mean, pseudocount=0.5) ---
ctrl_mean = X_broad_raw[is_control].mean(axis=0) + 0.5
treat_mean = X_broad_raw[is_treatment].mean(axis=0) + 0.5
log2fc = np.log2(treat_mean / ctrl_mean)

log2fc_df = pd.DataFrame({'gene_id': broad_gene_ids, 'log2FC': log2fc})
print(f"log2FC range: [{log2fc.min():.2f}, {log2fc.max():.2f}]")
print(f"Upregulated (log2FC > 0): {(log2fc > 0).sum()}, Downregulated: {(log2fc < 0).sum()}")

Broad signature genes: 1484
log2FC range: [-0.88, 1.45]
Upregulated (log2FC > 0): 483, Downregulated: 1001


In [24]:
# --- Plot 2: Clustered heatmap (1,400 genes × 10 samples) ---
from scipy.cluster.hierarchy import linkage, leaves_list

# Sort samples: controls first, then treatments grouped by sample_type
sample_order_df = sub[['sample', 'sample_type']].copy()
sample_order_df['_ctrl'] = (sample_order_df['sample_type'] == 'Control').astype(int)
sample_order_df = sample_order_df.sort_values(['_ctrl', 'sample_type'], ascending=[False, True])
ordered_samples = sample_order_df['sample'].tolist()

# Build genes × samples matrix in sample order
heatmap_broad = pd.DataFrame(
    X_broad,
    index=sub['sample'].values,
    columns=broad_gene_ids,
).T[ordered_samples]   # shape: 1400 × 10

# Hierarchical clustering on rows (genes) — ward linkage on standardized rows
Z = linkage(heatmap_broad.values, method='ward', metric='euclidean')
row_order = leaves_list(Z)
heatmap_sorted = heatmap_broad.iloc[row_order]

# Y-axis: show every 100th gene index, blank otherwise
n_genes = len(heatmap_sorted)
y_tickvals = list(range(0, n_genes, 100))
y_ticktext = [str(i) for i in y_tickvals]

sample_type_map = sub.set_index('sample')['sample_type'].to_dict()
x_labels = [f"{s}<br><i>{sample_type_map[s]}</i>" for s in ordered_samples]

fig = go.Figure(go.Heatmap(
    z=heatmap_sorted.values,
    x=x_labels,
    y=list(range(n_genes)),
    colorscale='RdBu_r',
    zmid=0,
    zmin=-3, zmax=3,
    colorbar=dict(title='Standardized<br>expression (z-score)', thickness=18),
))
fig.update_layout(
    title=dict(
        text=(f'Clustered Expression Heatmap — 1,400-Gene NSC Broad Signature<br>'
              f'<sup>Rows: genes sorted by hierarchical clustering (Ward linkage) | '
              f'Columns: samples, Controls grouped left</sup>'),
        x=0.5,
    ),
    xaxis_title='Sample',
    yaxis=dict(
        title='Gene rank (hierarchical order)',
        tickmode='array',
        tickvals=y_tickvals,
        ticktext=y_ticktext,
    ),
    xaxis=dict(tickangle=-35),
    height=800, width=950,
    template='plotly_white',
)
fig.show()
print("Heatmap computed. Row ordering by Ward-linkage dendrogram applied.")

Heatmap computed. Row ordering by Ward-linkage dendrogram applied.


In [25]:
# --- Plot 3: Open Targets therapeutic area summary for the 1,400-gene set ---
broad_sig_set = set(broad_gene_ids)

# Merge disease associations with the 1,400-gene set
broad_disease = disease_associations[disease_associations['gene_id'].isin(broad_sig_set)].copy()
print(f"Broad-signature genes with OT associations: {broad_disease['gene_id'].nunique()} / {len(broad_gene_ids)}")

# Explode therapeutic_areas (stored as list per row)
broad_ta = (broad_disease
            .dropna(subset=['therapeutic_areas'])
            .explode('therapeutic_areas')
            .dropna(subset=['therapeutic_areas']))

ta_counts = (broad_ta
             .groupby('therapeutic_areas')['gene_id']
             .nunique()
             .reset_index()
             .rename(columns={'gene_id': 'n_genes'})
             .sort_values('n_genes', ascending=True))  # ascending for horizontal bar

fig = go.Figure(go.Bar(
    x=ta_counts['n_genes'],
    y=ta_counts['therapeutic_areas'],
    orientation='h',
    marker_color='mediumslateblue',
    text=ta_counts['n_genes'],
    textposition='outside',
))
fig.update_layout(
    title=dict(
        text=(f'Open Targets Therapeutic Area Enrichment — 1,400-Gene NSC Broad Signature<br>'
              f'<sup>Count of unique stable genes linked to each therapeutic area (Open Targets associations)</sup>'),
        x=0.5,
    ),
    xaxis_title='Number of stable genes (unique)',
    yaxis_title='Therapeutic area',
    template='plotly_white',
    height=max(500, 28 * len(ta_counts)),
    width=950,
    margin=dict(l=280),
)
fig.show()

Broad-signature genes with OT associations: 1484 / 1484


In [26]:
# --- Plot 4: Volcano plot — 1,400 stable genes in context of all detected genes ---
from scipy.stats import ttest_ind

# Compute log2FC and t-test p-value for ALL genes in gene_cols (18,845)
all_raw = sub[gene_cols].values.astype(float)
ctrl_expr = all_raw[is_control]    # (2, 18845)
treat_expr = all_raw[is_treatment] # (8, 18845)

ctrl_mean_all  = ctrl_expr.mean(axis=0) + 0.5
treat_mean_all = treat_expr.mean(axis=0) + 0.5
log2fc_all = np.log2(treat_mean_all / ctrl_mean_all)

# Per-gene Welch's t-test (ttest_ind axis=0 over samples)
t_stat, p_val = ttest_ind(treat_expr, ctrl_expr, axis=0, equal_var=False)
p_val = np.where(p_val == 0, 1e-300, p_val)   # guard against exact-zero floats
neg_log10_p = -np.log10(p_val)

# Save per-gene stats tables for reuse
all_gene_stats = pd.DataFrame({
    'gene_id': gene_cols,
    'log2FC': log2fc_all,
    't_stat': t_stat,
    'p_value': p_val,
    'neg_log10_p': neg_log10_p,
})

is_stable_mask = np.isin(gene_cols, broad_gene_ids)
gray_df   = all_gene_stats[~is_stable_mask]
stable_df = all_gene_stats[is_stable_mask]

print(f"All detected genes: {len(all_gene_stats)}")
print(f"Stable (purple): {is_stable_mask.sum()}")

fig = go.Figure()
fig.add_trace(go.Scattergl(
    x=gray_df['log2FC'], y=gray_df['neg_log10_p'],
    mode='markers',
    marker=dict(color='lightgray', size=3, opacity=0.5),
    name='Other genes (n={:,})'.format(len(gray_df)),
    hovertemplate='%{customdata}<br>log2FC=%{x:.2f}<br>-log10(p)=%{y:.2f}<extra></extra>',
    customdata=gray_df['gene_id'],
))
fig.add_trace(go.Scattergl(
    x=stable_df['log2FC'], y=stable_df['neg_log10_p'],
    mode='markers',
    marker=dict(color='mediumpurple', size=5, opacity=0.8),
    name='NSC stable genes (n={:,})'.format(len(stable_df)),
    hovertemplate='%{customdata}<br>log2FC=%{x:.2f}<br>-log10(p)=%{y:.2f}<extra></extra>',
    customdata=stable_df['gene_id'],
))
fig.add_vline(x=0, line_width=1, line_dash='dash', line_color='black')

fig.update_layout(
    title=dict(
        text=(f'Volcano Plot — NSC Broad Signature (1,400 genes) in Context<br>'
              f'<sup>Treatment vs Control | Welch\'s t-test | '
              f'Purple = 100%-stable at NSC threshold=0.5</sup>'),
        x=0.5,
    ),
    xaxis_title='log₂ Fold Change (treatment / control)',
    yaxis_title='−log₁₀(p-value)',
    legend=dict(x=0.02, y=0.98, bgcolor='rgba(255,255,255,0.8)'),
    template='plotly_white',
    width=950, height=600,
)
fig.show()

All detected genes: 18845
Stable (purple): 1484


1. The volcano plot — what it shows:
A volcano plot puts two pieces of information together for every gene:

X-axis: log2 fold change — how much did expression change? (large positive = big increase, large negative = big decrease, near 0 = no change)
Y-axis: -log10(p-value) — how confident are we the change isn't random? (higher = more significant)

The negative log10 flips the scale so that small p-values (significant) appear at the top. So:

Top of plot = highly statistically significant
Far left/right = large effect size
Top corners = both significant AND large effect — the "interesting" genes
Bottom middle = uninteresting (no change, not significant)
Bottom corners = large effect but no statistical confidence (often noise)

In [ ]:
# --- Save outputs to GCS ---

# # 1. Broad signature gene list (the 1,400 genes)
# nsc_broad_signature = broad_sig_df[['gene_id', 'selection_frequency']].copy()
# nsc_broad_signature.to_csv('gs://gene_datasets/nsc_broad_signature.csv', index=False)
# print(f"Saved nsc_broad_signature.csv ({len(nsc_broad_signature)} genes)")

# # 2. log2FC per gene (broad signature)
# log2fc_df.to_csv('gs://gene_datasets/nsc_broad_log2fc.csv', index=False)
# print(f"Saved nsc_broad_log2fc.csv ({len(log2fc_df)} genes)")

# # 3. Full per-gene stats (all 18,845 genes: log2FC, t-stat, p-value)
# all_gene_stats.to_csv('gs://gene_datasets/nsc_all_gene_stats.csv', index=False)
# print(f"Saved nsc_all_gene_stats.csv ({len(all_gene_stats)} genes)")

# print("\nAll outputs saved to gs://gene_datasets/")